In [20]:
#converting long dataset into chunks (enough samples for model to get trained)
#model used:DNABERT 2 model

In [21]:
#storing sequences
from Bio import SeqIO
def read_fasta(file):
    return [str(record.seq) for record in SeqIO.parse(file,"fasta")]
meca_seq=read_fasta(r"C:\Users\Amishi Singh\Desktop\BioEndsem\mecA_comp\mecA_complete_clean.fasta")
mecc_seq=read_fasta(r"C:\Users\Amishi Singh\Desktop\BioEndsem\mecC\mecC_clean.fasta")
mssa_seq=read_fasta(r"C:\Users\Amishi Singh\Desktop\BioEndsem\mssa\mssa_clean.fasta")

In [22]:
#split into chunks
def split_seq(seq,chunk_size=300):
    return [
        seq[i:i+chunk_size] for i in range(0,len(seq),chunk_size) if(len(seq[i:i+chunk_size])==chunk_size)
    ]


In [23]:
meca_chunks=[]
for seq in meca_seq: 
   meca_chunks.extend(split_seq(seq))

mecc_chunks=[]
for seq in mecc_seq: 
   mecc_chunks.extend(split_seq(seq))

mssa_chunks=[]
for seq in mssa_seq: 
   mssa_chunks.extend(split_seq(seq))
print(len(meca_chunks[0]))

300


In [24]:
#creating labels
import pandas as pd
mrsa_chunks=meca_chunks+mecc_chunks
df=pd.DataFrame({
    "sequence":mrsa_chunks+mssa_chunks,
    "label":[1]*len(mrsa_chunks)+[0]*len(mssa_chunks)
})


In [25]:
#balancing dataset
min_size=min(len(mrsa_chunks),len(mssa_chunks))

df_mrsa=df[df.label==1].sample(min_size)
df_mssa=df[df.label==0].sample(min_size)
#mixes random rows together and then reset index
df=pd.concat([df_mrsa,df_mssa]).sample(frac=1).reset_index(drop=True)


In [26]:
#creating the dataset and then splitting into test and training part 
from datasets import Dataset
dataset=Dataset.from_pandas(df)
dataset=dataset.train_test_split(test_size=0.2)

In [27]:
#making kmers 
def to_kmers(seq,k=6):
    return "".join([seq[i:i+k] for i in range (len(seq)-k+1)])
#sequence field is now replaced by kmers of len 6
dataset=dataset.map(lambda x:{"sequence":to_kmers(x["sequence"])})


Map:   0%|          | 0/436947 [00:00<?, ? examples/s]

Map:   0%|          | 0/109237 [00:00<?, ? examples/s]

In [28]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification
#name of the model
model_name="zhihan1996/DNA_bert_6"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    trust_remote_code=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

DNABertForSequenceClassification LOAD REPORT from: zhihan1996/DNA_bert_6
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [29]:
def tokenize(example):
    return tokenizer(
        example["sequence"], #input data
        truncation=True,#if input text>limit cut off tokens
        padding="max_length",#every output of eq len
        max_length=512
    )
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/436947 [00:00<?, ? examples/s]

Map:   0%|          | 0/109237 [00:00<?, ? examples/s]

In [30]:
#removing unused text column
dataset=dataset.remove_columns(["sequence"])
dataset.set_format("torch")

In [31]:
import sys
!{sys.executable} -m pip install transformers

'c:\Users\Amishi' is not recognized as an internal or external command,
operable program or batch file.


In [32]:
from transformers import TrainingArguments
training_args=TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [33]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [34]:

from transformers import Trainer
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

In [35]:
results = trainer.evaluate()
print(results)

KeyboardInterrupt: 